# Mini-Wikipedia RAG Ingestion with LlamaIndex and ChromaDB

This notebook builds the ingestion side of your AI system. It reads PDF files from `data/wikipedia`, splits the text into smaller chunks, embeds those chunks with a local Hugging Face model, and stores them in a persistent ChromaDB collection.

The flow is:

1. Install and import the required libraries.
2. Point the notebook at your PDF folder and ChromaDB folder.
3. Load PDFs as LlamaIndex documents.
4. Chunk the documents into retrieval-friendly nodes.
5. Insert the nodes into ChromaDB.
6. Run a small retrieval test to confirm the vector database is working.

In [ ]:
# Install the RAG ingestion dependencies in the active notebook kernel.
# Run this cell once, then restart the kernel if imports still fail.
%pip install -U llama-index llama-index-vector-stores-chroma llama-index-embeddings-huggingface llama-index-readers-file chromadb pypdf sentence-transformers

In [2]:
import re
import unicodedata
from pathlib import Path

import chromadb
from chromadb.errors import NotFoundError
from pypdf import PdfReader
from llama_index.core import Settings, StorageContext, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import Document, TextNode
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore

## 1. Configure Paths and Embeddings

The PDF folder is your source corpus. The ChromaDB folder is where the vector database will be saved, so you can close the notebook and load the same collection later.

This notebook uses `BAAI/bge-small-en-v1.5` because it is a strong local embedding model and does not require an API key.

In [3]:
PROJECT_ROOT = Path(r"C:\Program Files\Studying\coding\RAG_project")
PDF_DIR = PROJECT_ROOT / "data" / "wikipedia"
CHROMA_DIR = PROJECT_ROOT / "chromadb_store"
COLLECTION_NAME = "mini_wikipedia"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 120
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"

if not PDF_DIR.exists():
    raise FileNotFoundError(f"PDF folder not found: {PDF_DIR}")

# LlamaIndex uses Settings as global defaults for later index and query operations.
Settings.embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL_NAME)
Settings.llm = None

print(f"PDF source folder: {PDF_DIR}")
print(f"ChromaDB folder: {CHROMA_DIR}")
print(f"Embedding model: {EMBED_MODEL_NAME}")

LLM is explicitly disabled. Using MockLLM.
PDF source folder: C:\Program Files\Studying\coding\RAG_project\data\wikipedia
ChromaDB folder: C:\Program Files\Studying\coding\RAG_project\chromadb_store
Embedding model: BAAI/bge-small-en-v1.5


## 2. Load PDF Documents

PDF text extraction is often the weakest part of a RAG pipeline. Some PDFs contain real text and extract cleanly, while others are scanned images or have poor internal encoding.

This notebook now loads PDFs page by page with `pypdf`, normalizes common extraction artifacts, and prints a warning when a page returns no usable text. If the text is still unreadable after cleanup, that PDF likely needs OCR before chunking.

In [4]:
def list_pdf_files(pdf_dir: Path) -> list[Path]:
    """Return all PDF files under the source folder."""
    # Use rglob so nested Wikipedia folders are included automatically.
    return sorted(pdf_dir.rglob("*.pdf"))


def normalize_extracted_text(text: str) -> str:
    """Clean common PDF extraction artifacts while preserving readable content."""
    # Unicode normalization fixes some ligatures and inconsistent character forms.
    cleaned_text = unicodedata.normalize("NFKC", text or "")
    cleaned_text = cleaned_text.replace("\x00", " ").replace("\u00ad", "")

    # Join words broken across line endings, then normalize whitespace.
    cleaned_text = re.sub(r"(\w)-\n(\w)", r"\1\2", cleaned_text)
    cleaned_text = re.sub(r"\r\n?", "\n", cleaned_text)
    cleaned_text = re.sub(r"[ \t]+", " ", cleaned_text)
    cleaned_text = re.sub(r"\n{3,}", "\n\n", cleaned_text)

    return cleaned_text.strip()


def load_pdf_documents(pdf_dir: Path) -> list[Document]:
    """Load PDF pages into LlamaIndex Documents with cleaned text and metadata."""
    documents: list[Document] = []
    empty_pages: list[tuple[str, int]] = []

    for pdf_path in list_pdf_files(pdf_dir):
        reader = PdfReader(str(pdf_path))

        # Extract page by page so each document keeps an exact page reference.
        for page_number, page in enumerate(reader.pages, start=1):
            raw_text = page.extract_text() or ""
            cleaned_text = normalize_extracted_text(raw_text)

            if not cleaned_text:
                empty_pages.append((pdf_path.name, page_number))
                continue

            documents.append(
                Document(
                    text=cleaned_text,
                    metadata={
                        "source_type": "wikipedia_pdf",
                        "file_name": pdf_path.name,
                        "file_path": str(pdf_path),
                        "page": page_number,
                        "page_label": str(page_number),
                    },
                )
            )

    if empty_pages:
        print("Warning: some PDF pages returned no readable text.")
        for file_name, page_number in empty_pages[:10]:
            print(f"- {file_name}, page {page_number}")
        if len(empty_pages) > 10:
            print(f"... and {len(empty_pages) - 10} more empty page(s)")
        print("If the extracted text still looks messy, those PDFs may require OCR before RAG ingestion.")

    return documents


def preview_pdf_extraction(documents: list[Document], preview_count: int = 3) -> None:
    """Print a few extracted pages so you can quickly judge extraction quality."""
    for index, document in enumerate(documents[:preview_count], start=1):
        print(f"\n--- Preview {index} ---")
        print(f"Source: {document.metadata['file_name']} | page: {document.metadata['page']}")
        print(document.text[:1200])


pdf_files = list_pdf_files(PDF_DIR)
print(f"Found {len(pdf_files)} PDF file(s).")
for pdf_file in pdf_files[:10]:
    print(f"- {pdf_file.name}")

if len(pdf_files) > 10:
    print(f"... and {len(pdf_files) - 10} more")


documents = load_pdf_documents(PDF_DIR)
print(f"Loaded {len(documents)} page document(s) from PDFs.")
preview_pdf_extraction(documents)

Found 37 PDF file(s).
- Artificial intelligence.pdf
- Carrie Lam.pdf
- China.pdf
- Chinese University of Hong Kong.pdf
- City University of Hong Kong.pdf
- Computer vision.pdf
- Data science.pdf
- Deep learning.pdf
- Donald Tsang.pdf
- Education University of Hong Kong.pdf
... and 27 more
Loaded 836 page document(s) from PDFs.

--- Preview 1 ---
Source: Artificial intelligence.pdf | page: 1
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically
associated with human intelligence, such as learning, reasoning, problem-solving, perception, and
decision-making. It is a field of research in engineering, mathematics and computer science that
develops and studies methods and software that enable machines to perceive their environment
and use learning and intelligence to take actions that maximize their chances of achieving defined
goals.
High-profile applications of AI include advanced web search engines, chatbots, virtual assistants,
autonomous veh

## 3. Chunk the Text

RAG works better when documents are split into chunks that are large enough to contain meaning but small enough to retrieve precisely. Here we use sentence-aware chunking with overlap, so nearby context is not lost at chunk boundaries.

In [5]:
def build_text_nodes(
    documents: list[Document],
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
) -> list[TextNode]:
    """Split loaded documents into chunks that can be embedded and retrieved."""
    # SentenceSplitter tries to preserve sentence boundaries instead of cutting text blindly.
    splitter = SentenceSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    nodes = splitter.get_nodes_from_documents(documents)

    # Store a readable chunk number in metadata to make retrieved results easier to inspect.
    for chunk_number, node in enumerate(nodes, start=1):
        node.metadata["chunk_number"] = chunk_number
        node.metadata["chunk_size"] = chunk_size
        node.metadata["chunk_overlap"] = chunk_overlap

    return nodes


documents = load_pdf_documents(PDF_DIR)
nodes = build_text_nodes(documents)

print(f"Loaded {len(documents)} document unit(s) from PDFs.")
print(f"Created {len(nodes)} text chunk(s).")
print("\nPreview of first chunk:")
print(nodes[0].get_content()[:800] if nodes else "No chunks created.")

Loaded 836 document unit(s) from PDFs.
Created 836 text chunk(s).

Preview of first chunk:
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically
associated with human intelligence, such as learning, reasoning, problem-solving, perception, and
decision-making. It is a field of research in engineering, mathematics and computer science that
develops and studies methods and software that enable machines to perceive their environment
and use learning and intelligence to take actions that maximize their chances of achieving defined
goals.
High-profile applications of AI include advanced web search engines, chatbots, virtual assistants,
autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the
2020s, generative AI has become widely available to generate images, audio, and videos from text
prompts.
The tradi


In [6]:
nodes[0].get_content()

"Artificial intelligence (AI) is the capability of computational systems to perform tasks typically\nassociated with human intelligence, such as learning, reasoning, problem-solving, perception, and\ndecision-making. It is a field of research in engineering, mathematics and computer science that\ndevelops and studies methods and software that enable machines to perceive their environment\nand use learning and intelligence to take actions that maximize their chances of achieving defined\ngoals.\nHigh-profile applications of AI include advanced web search engines, chatbots, virtual assistants,\nautonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the\n2020s, generative AI has become widely available to generate images, audio, and videos from text\nprompts.\nThe traditional goals of AI research include learning, reasoning, knowledge representation, planning,\nnatural language processing, and perception, as well as support for robotics. To reach these go

In [7]:
nodes[1].get_content()

'Goals\nThe general problem of simulating (or creating) intelligence has been broken into subproblems.\nThese consist of particular traits or capabilities that researchers expect an intelligent system to\ndisplay. The traits described below have received the most attention and cover the scope of AI\nresearch.\nReasoning and problem-solving\nEarly researchers developed algorithms that imitated step-by-step reasoning that humans use when\nthey solve puzzles or make logical deductions. By the late 1980s and 1990s, methods were\ndeveloped for dealing with uncertain or incomplete information, employing concepts from probability\nand economics.\nMany of these algorithms are insufficient for solving large reasoning problems because they\nexperience a "combinatorial explosion": They become exponentially slower as the problems grow.\nEven humans rarely use the step-by-step deduction that early AI research could model. Humans\nsolve most of their problems using fast, intuitive judgments.\nReason

In [8]:
nodes[2].get_content()

'A knowledge base is a body of knowledge represented in a form that can be used by a program. An\nontology is the set of objects, relations, concepts, and properties used by a particular domain of\nknowledge. Knowledge bases need to represent things such as objects, properties, categories, and\nrelations between objects; situations, events, states, and time; causes and effects; knowledge about\nknowledge (what we know about what other people know); default reasoning (things that humans\nassume are true until they are told differently and will remain true even when other facts are\nchanging); and many other aspects and domains of knowledge.\nAmong the most difficult problems in knowledge representation are the breadth of commonsense\nknowledge (the set of atomic facts that the average person knows is enormous); and the\nsub-symbolic form of most commonsense knowledge (much of what people know is not represented\nas "facts" or "statements" that they could express verbally). There is also

In [9]:
print(len(nodes[0].get_content()))
print(len(nodes[1].get_content()))
print(len(nodes[2].get_content()))

2307
1587
2252


## 4. Insert Chunks into ChromaDB

This cell creates a persistent ChromaDB collection and inserts the chunks through LlamaIndex. `RESET_COLLECTION = True` rebuilds this notebook's collection cleanly, which prevents duplicate chunks when you rerun the notebook. Set it to `False` only when you intentionally want to append new chunks.

In [7]:
RESET_COLLECTION = True


def get_chroma_collection(
    chroma_dir: Path,
    collection_name: str,
    reset_collection: bool = False,
):
    """Create or load a persistent ChromaDB collection."""
    # PersistentClient writes vectors and metadata to disk instead of keeping them in memory.
    chroma_dir.mkdir(parents=True, exist_ok=True)
    client = chromadb.PersistentClient(path=str(chroma_dir))

    # Some ChromaDB versions raise NotFoundError when the collection is missing,
    # while older versions may raise ValueError. Catch both so reset stays safe.
    if reset_collection:
        try:
            client.delete_collection(collection_name)
            print(f"Deleted existing collection: {collection_name}")
        except (NotFoundError, ValueError):
            print(f"Collection did not exist yet: {collection_name}")

    return client.get_or_create_collection(collection_name)


def build_or_update_vector_index(
    text_nodes: list[TextNode],
    chroma_dir: Path = CHROMA_DIR,
    collection_name: str = COLLECTION_NAME,
    reset_collection: bool = RESET_COLLECTION,
) -> VectorStoreIndex:
    """Embed nodes and store them in ChromaDB through LlamaIndex."""
    # ChromaVectorStore adapts the Chroma collection to LlamaIndex's vector store interface.
    collection = get_chroma_collection(chroma_dir, collection_name, reset_collection)
    vector_store = ChromaVectorStore(chroma_collection=collection)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)

    # VectorStoreIndex computes embeddings for each node and writes vectors plus metadata.
    return VectorStoreIndex(text_nodes, storage_context=storage_context)


index = build_or_update_vector_index(nodes)
collection = get_chroma_collection(CHROMA_DIR, COLLECTION_NAME)

print(f"Collection name: {COLLECTION_NAME}")
print(f"Stored vector count: {collection.count()}")

Collection did not exist yet: mini_wikipedia
Collection name: mini_wikipedia
Stored vector count: 836


## 5. Query the Stored Chunks

Once the chunks are inside ChromaDB, you do not need to insert them again every time you reopen the notebook.

After reopening the notebook, rerun the setup cells so the imports, paths, and embedding model are available, then load the existing Chroma collection from disk and query it. The code cell below supports both cases:

- querying the freshly built index in the current session
- reconnecting to an existing persisted ChromaDB collection after reopening the notebook

In [10]:
def load_existing_index(
    chroma_dir: Path = CHROMA_DIR,
    collection_name: str = COLLECTION_NAME,
) -> VectorStoreIndex:
    """Reconnect to an existing persisted ChromaDB collection without reinserting data."""
    # This is the path you use after closing and reopening the notebook.
    collection = get_chroma_collection(chroma_dir, collection_name, reset_collection=False)
    vector_store = ChromaVectorStore(chroma_collection=collection)
    return VectorStoreIndex.from_vector_store(vector_store=vector_store, embed_model=Settings.embed_model)


def retrieve_relevant_chunks(
    query: str,
    top_k: int = 5,
    query_index: VectorStoreIndex | None = None,
):
    """Retrieve the most relevant chunks for a user question."""
    # Use the provided index when one is passed in, otherwise fall back to the current session index.
    active_index = query_index if query_index is not None else index
    retriever = active_index.as_retriever(similarity_top_k=top_k)
    results = retriever.retrieve(query)

    # Print source metadata beside each chunk so you can inspect where answers come from.
    for rank, result in enumerate(results, start=1):
        metadata = result.node.metadata
        file_name = metadata.get("file_name", "unknown file")
        page_label = metadata.get("page_label", metadata.get("page", "unknown page"))
        chunk_number = metadata.get("chunk_number", "unknown chunk")

        print(f"\n--- Result {rank} | score={result.score:.4f} ---")
        print(f"Source: {file_name} | page: {page_label} | chunk: {chunk_number}")
        print(result.node.get_content()[:1200])

    return results


# Use the in-memory index if you just inserted data in this session.
sample_query = "CUHK and HKU are the best schools in Hong Kong. Which one is better?"
retrieved_chunks = retrieve_relevant_chunks(sample_query, top_k=3)


--- Result 1 | score=0.6716 ---
Source: Chinese University of Hong Kong.pdf | page: 11 | chunk: 107
CUHK has been consistently regarded as one of the top two universities in Hong Kong by various
university rankings, with the other being the University of Hong Kong.
Overall rankings
CUHK was ranked #32 worldwide in QS WUR 2026, #44 worldwide in THE WUR 2025, and #37
worldwide in US News & Report 2025–2026, and #101-150 worldwide in ARWU 2024. CUHK has
been the top Hong Kong institution in the ARWU, which is based on awards and research output,
including those league tables in 2006, 2010, 2011, and 2013. The HKU Public Opinion Programme
survey (2012) gave it the 2nd place. China's Alumni Association placed it among the "6-Star Greater
China's Universities" (the highest level). It was ranked fourth in the association's 2014 Ranking of
Institutions with the Most Best Disciplines in HK, Macau and Taiwan.
CUHK was 65th worldwide in terms of aggregate performance across THE, QS, and ARWU, as

In [ ]:
# If you reopen the notebook later, rerun the setup cells and then use this block.
reloaded_index = load_existing_index()
retrieved_chunks_after_reopen = retrieve_relevant_chunks(sample_query, top_k=3, query_index=reloaded_index)

## 6. Optional: Build a Query Engine Later

The notebook above completes the vector database insertion step. When you are ready to generate natural-language answers, connect an LLM and create a query engine from the same index.

For example, you can later use a local Ollama model, OpenAI, Azure OpenAI, or another LlamaIndex-supported LLM. Retrieval is already working once the previous cell returns relevant chunks.